# Vyhodnotenie uzávierok — model vs. realita

**Hlavná otázka:** keď model povie „zavretie tejto ulice presunie dopravu na X",
zodpovedá to tomu, čo sa reálne stalo?

Porovnávame s dvomi nezávislými zdrojmi:

| zdroj | čo hovorí | sila |
|---|---|---|
| **oficiálna obchádzka** (NDIC) | kadiaľ úrad dopravu poslal | priame, rovnaká veličina — *kam sa presunie doprava* |
| **Waze zápchy** | kde sa reálne zhoršila situácia | nepriame — zdržanie ≠ objem, len poradie |

Prvý zdroj je primárny. Je dostupný pre každú uzávierku a nepotrebuje prevod
medzi zdržaním a objemom.

### Dve veci, ktoré sa oproti prvej verzii zmenili

1. **Uzávierka sa kotví cez `segment_id`, nie cez názov ulice.** Pole
   `street_name` v NDIC nehovorí, ktorý úsek je zavretý — napr. záznam
   „VMO Dobrovského" popisuje *obchádzku* cez Provazníkovu, záznam „Benešova"
   sa v skutočnosti týka Minoritskej. `segment_id` ukazuje na presné miesto.
2. **Rozlišuje sa úplné uzavretie od zúženia.** Väčšina záznamov sú
   `roadworks` — často „úplná uzavírka **jízdního pruhu**", nie celej ulice.

## 1 · Nastavenie

In [1]:
# ═══ NASTAVENIE ═══════════════════════════════════════════════════════
API_URL   = "http://localhost:8000"
CSV_DIR   = "../data/data_vyhodnotenie_brno"
VARIANT   = "updated_version_12"

RADIUS_M       = 400     # dokedy od kotvy segmentu hľadať linky uzávierky
TOP_N          = 10      # koľko ulíc ukazujeme v tabuľkách
OKNO_PRED_DNI  = 14
NABEH_DNI      = 2       # prvé dni sa vynechajú, vodiči sa učia
OKNO_POCAS_MAX = 28
MIN_ZAPCH      = 3
QUALITY_MIN    = 50
# ══════════════════════════════════════════════════════════════════════

import json, math, re, time, unicodedata, urllib.parse, urllib.request
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

CSV = Path(CSV_DIR)
OUT = Path("..") / "simulation_for_article" / VARIANT / "vyhodnotenie"
OUT.mkdir(parents=True, exist_ok=True)

def api(path, **params):
    url = f"{API_URL}{path}" + ("?" + urllib.parse.urlencode(params) if params else "")
    with urllib.request.urlopen(url, timeout=900) as r:
        return json.load(r)

try:
    api("/api/reports/od-summary"); print("API beží ✓")
except Exception as e:
    print("API NEDOSTUPNÉ:", e)

def norm(s):
    if not isinstance(s, str):
        return ""
    return "".join(c for c in unicodedata.normalize("NFKD", s.lower())
                   if not unicodedata.combining(c)).strip()

def vzdial_m(lat1, lon1, lat2, lon2):
    """Rovinná aproximácia, na úrovni mesta postačuje."""
    return math.hypot((lon2 - lon1) * 73_000, (lat2 - lat1) * 111_000)

API beží ✓


## 2 · Dáta

In [2]:
from shapely import wkb as _wkb
NULLS = ["NULL", "null", "\\N"]

def geom(h):
    if not isinstance(h, str) or not h.strip():
        return None
    try:
        return _wkb.loads(h, hex=True)
    except Exception:
        return None

jams = pd.read_csv(CSV / "jams.csv", na_values=NULLS, low_memory=False,
                   usecols=["id","first_seen","street_name","segment_id","delay_seconds",
                            "speed_kmh","speed_normal_kmh","quality_score"])
jams["ts"] = pd.to_datetime(jams.first_seen, errors="coerce", utc=True)
# Waze píše časy v UTC; bez prevodu by víkendový filter chytal nesprávne dni.
jams["ts_local"] = jams.ts.dt.tz_convert("Europe/Prague")
jams["den"] = jams.ts_local.dt.date
jams["vsedny"] = jams.ts_local.dt.weekday < 5

segs = pd.read_csv(CSV / "road_segments.csv", na_values=NULLS, low_memory=False)
segs["stred"] = segs.geog.apply(lambda g: (lambda x: (x.centroid.y, x.centroid.x)
                                           if x is not None else None)(geom(g)))

restr = pd.read_csv(CSV / "restrictions.csv", na_values=NULLS, low_memory=False)
restr["zdroj"] = restr.external_ids.fillna("").apply(
    lambda s: "ndic" if '"ndic"' in s else ("waze" if '"waze"' in s else "ine"))
for c in ("valid_from", "valid_to"):
    restr[c] = pd.to_datetime(restr[c], errors="coerce", utc=True)

print(f"jams   {len(jams):>8,}  {jams.ts_local.min():%Y-%m-%d} → {jams.ts_local.max():%Y-%m-%d}")
print(f"segs   {len(segs):>8,}  s geometriou {segs.stred.notna().sum():,}")
print(f"NDIC   {int((restr.zdroj=='ndic').sum()):>8,} záznamov")

jams    283,869  2026-04-14 → 2026-08-28
segs     25,923  s geometriou 25,923
NDIC        122 záznamov


## 3 · Sieť modelu

In [3]:
_gj = api("/api/links", min_volume=0)
LINKY = []
for f in _gj["features"]:
    p, g = f["properties"], (f.get("geometry") or {})
    cs = g.get("coordinates") or []
    if g.get("type") == "MultiLineString":
        cs = [c for part in cs for c in part]
    if not cs:
        continue
    lats = [c[1] for c in cs]; lons = [c[0] for c in cs]
    LINKY.append({"link_id": int(p["link_id"]), "name": p.get("name"),
                  "ref": str(p.get("osm_ref") or ""), "typ": p.get("link_type"),
                  "lanes": p.get("lanes"), "vol": float(p.get("wd_daily_tot") or 0),
                  "coords": cs, "stred": (sum(lats)/len(lats), sum(lons)/len(lons))})
MENA_MODELU = {l["name"] for l in LINKY if l["name"]}
print(f"liniek {len(LINKY):,} | pomenovaných ulíc {len(MENA_MODELU):,}")

liniek 36,476 | pomenovaných ulíc 2,060


## 4 · Oficiálne obchádzky z popisov NDIC

Popisy majú ustálený tvar: *„Objízdná trasa … je navržena od křižovatky ul. A
x ul. B po ul. C, dále na ul. D"*. Vytiahneme z nich názvy ulíc a necháme len
tie, ktoré existujú v sieti modelu — to zároveň odfiltruje chybné výsledky
regexu.

In [4]:
from __future__ import annotations

import re

_MARKERY = ("Objízdná", "Objížďka", "objízdn", "Objízdné")

# Klauzuly, ktoré menujú križovatku, nie trasu. Vyhodíme ich pred extrakciou.
_KRIZOVATKA = re.compile(
    r"(?:od|ke|na|u)\s+křižovatk\w*\s+(?:ulic\w*\s+)?"
    r"[^,;.]*?(?:\s+x\s+[^,;.]*?)+(?=[,;.]|\s+(?:dále|po|přes|pro)\b|$)",
    re.IGNORECASE)

# "ul. Nádražní" / "ulice třída Generála Píky" / "ulici Koliště"
_PREFIX = r"(?:ul\.|ulice|ulic|ulici)"
_MENO = (r"(?:třída|náměstí|nábřeží)?\s*"
         r"[A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ][\wÁ-Žá-ž.]*"
         r"(?:\s+[A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ][\wÁ-Žá-ž.]*){0,3}")

_PAT_PREFIX = re.compile(_PREFIX + r"\s+(" + _MENO + r")")
# pokračovanie zoznamu: ", Průmyslová, Ostravská" alebo " a Žarošická"
_PAT_ZOZNAM = re.compile(r"[,;]\s*(" + _MENO + r")|\s+a\s+(" + _MENO + r")")

_STOP_SLOVA = {
    "vydal", "magistrát", "krajský", "úřad", "brno", "brně", "termín", "objízdná",
    "objížďka", "trasa", "směr", "směru", "obec", "obce", "obcí", "mč", "pro",
    "dále", "přes", "silnice", "silnici", "místními", "komunikacemi", "podzemní",
    "parkování", "opačný", "pořadí", "divadlem", "janáčkovým",
}


def _cisty(meno: str) -> str:
    meno = meno.strip(" .,;:")
    # odsekni koncové slová, ktoré k názvu nepatria
    casti = meno.split()
    while casti and casti[-1].lower() in _STOP_SLOVA:
        casti.pop()
    return " ".join(casti)


def ulice_obchadzky(popis: str | None) -> list[str]:
    """Ulice obchádzky v poradí výskytu. Prázdny zoznam, ak popis obchádzku nemá."""
    if not isinstance(popis, str):
        return []

    zaciatky = [popis.find(m) for m in _MARKERY]
    zaciatky = [i for i in zaciatky if i >= 0]
    if not zaciatky:
        return []                      # (1) žiadna obchádzka -> nič, žiadny fallback
    usek = popis[min(zaciatky):]

    usek = _KRIZOVATKA.sub(" ", usek)  # (4) preč s križovatkami

    out: list[str] = []
    for m in _PAT_PREFIX.finditer(usek):
        meno = _cisty(m.group(1))
        if meno and meno not in out:
            out.append(meno)
        # (2)+(3) pokračovanie zoznamu hneď za nájdeným názvom
        chvost = usek[m.end():m.end() + 120]
        for z in _PAT_ZOZNAM.finditer(chvost):
            dalsi = _cisty(z.group(1) or z.group(2) or "")
            if not dalsi:
                break
            if dalsi not in out:
                out.append(dalsi)
    return out

def obchadzka_v_modeli(popis):
    """Len ulice, ktoré sieť modelu pozná — inak sa nedá spočítať poradie."""
    return [u for u in ulice_obchadzky(popis) if u in MENA_MODELU]

ndic = restr[restr.zdroj == "ndic"].copy()
ndic["obchadzka"] = ndic.description_cs.apply(obchadzka_v_modeli)
ndic["n_obch"] = ndic.obchadzka.apply(len)
kand = ndic[(ndic.n_obch > 0) & ndic.segment_id.notna()].copy()

_ma_text = ndic.description_cs.fillna("").str.contains("Obj[íi]zdn|Objížďka", regex=True)
print(f"NDIC záznamov: {len(ndic)}")
print(f"  s obchádzkou v popise:            {int(_ma_text.sum())}")
print(f"  s obchádzkou napárovanou na sieť: {len(kand)}\n")
display(kand[["id","street_name","valid_from","valid_to","segment_id","obchadzka"]]
        .sort_values("valid_from").reset_index(drop=True))


NDIC záznamov: 122
  s obchádzkou v popise:            21
  s obchádzkou napárovanou na sieť: 18



,id,street_name,valid_from,valid_to,segment_id,obchadzka
0,94340,Ferrerova,2025-03-14 23:00:00+00:00,2026-07-31 21:59:00+00:00,17429.0,"[Charbulova, Elišky Krásnohorské]"
1,94377,Koperníkova,2025-10-05 22:00:00+00:00,2026-06-30 21:59:00+00:00,19763.0,"[Skopalíkova, Kosmákova]"
2,94342,Hlaváčova,2025-10-31 23:00:00+00:00,2026-05-24 21:59:00+00:00,20420.0,"[Rokytova, Žarošická, Jedovnická]"
3,94388,Kosmákova,2026-01-25 23:00:00+00:00,2026-06-30 21:59:00+00:00,19729.0,"[Koperníkova, Stará, Kosmákova]"
4,94402,Viniční,2026-02-09 07:00:00+00:00,2026-12-13 22:59:00+00:00,17933.0,"[Gajdošova, Škrochova]"
5,94415,Olomoucká,2026-02-27 23:00:00+00:00,2026-04-30 21:59:00+00:00,22803.0,"[Těžební, Průmyslová, Ostravská]"
6,94413,Skopalíkova,2026-02-28 23:00:00+00:00,2026-11-30 22:59:00+00:00,19711.0,"[Skopalíkova, Bubeníčkova, Koperníkova]"
7,94434,Vlárská,2026-03-13 07:06:00+00:00,2026-05-31 21:59:00+00:00,22319.0,[Vlárská]
8,94488,Moravské náměstí,2026-04-12 22:00:00+00:00,2026-05-17 21:59:00+00:00,16493.0,"[Kounicova, Koliště, Rooseveltova, Dvořákova, Jezuitská]"
9,94493,Jihlavská,2026-04-13 22:00:00+00:00,2026-04-30 21:59:00+00:00,8118.0,"[Kamenice, Akademická, Jihlavská, Osová, Pod Nemocnicí]"


## 5 · Kde presne uzávierka je

Kotva je `segment_id` → geometria v `road_segments`. Linky uzávierky sú tie,
ktoré ležia do `RADIUS_M` od kotvy a majú **rovnaké číslo cesty** (alebo,
ak segment číslo nemá, rovnaký názov). Bez tejto podmienky by buffer chytal
súbežné ulice — v prvej verzii takto pri VMO Dobrovského vypadol Husovický
tunel s 32 tis. voz/deň.

In [5]:
SEG = segs.set_index("id")

def linky_uzavierky(row):
    if pd.isna(row.segment_id) or int(row.segment_id) not in SEG.index:
        return [], "segment nenájdený"
    s = SEG.loc[int(row.segment_id)]
    if s.stred is None:
        return [], "segment bez geometrie"
    slat, slon = s.stred

    refy = {str(x).strip() for x in (s.road_ref, row.road_number)
            if isinstance(x, str) and str(x).strip()}
    meno = norm(s["name"]) if isinstance(s["name"], str) else ""

    blizke = [(vzdial_m(l["stred"][0], l["stred"][1], slat, slon), l) for l in LINKY]
    blizke = [(d, l) for d, l in blizke if d <= RADIUS_M]

    if refy:
        sel = [l for d, l in blizke if l["ref"] in refy]
        sposob = f"číslo cesty {'/'.join(sorted(refy))}"
    else:
        sel = [l for d, l in blizke if meno and norm(l["name"]) == meno]
        sposob = f"názov segmentu „{s['name']}“"

    if not sel:                       # posledná záchrana: najbližšie linky
        sel = [l for d, l in sorted(blizke)[:6]]
        sposob = "najbližšie linky (bez zhody čísla/názvu)"
    return sel, sposob

def typ_uzavierky(popis):
    """Rozsah uzávierky z popisu: úplné uzavretie vs. zúženie.

    Toto rozlíšenie je zásadné. Väčšina NDIC záznamov sú `roadworks`, kde
    cesta zostáva prejazdná — „částečná uzavírka … zachování průjezdu
    v režimu 1 + 1". Ak sa taká uzávierka modeluje ako úplné uzavretie,
    znemožní sa obchádzka, ktorá po tej ceste vedie.

    Zmerané na Jihlavskej (94493): predpísaná obchádzka je „Kamenice –
    Akademická – **Jihlavská** – Osová", teda po samotnej uzavretej ceste.
    Pri modelovaní ako úplné uzavretie skončila Osová na 2059. mieste
    z 2060; pri modelovaní ako zúženie na 2. mieste. Celkový rozsah zmeny
    klesol z 1,25 mil. na 186 tis. voz/deň, čo je vecne realistickejšie.
    """
    p = (popis or "").lower()
    CIASTOCNE = ("částečná uzavírka", "částečné uzavření", "režimu 1 + 1",
                 "režim 1 + 1", "režimu 1+1", "zachování průjezdu",
                 "jízdního pruhu", "jednoho pruhu", "odbočovací pruh",
                 "zúžení", "převeden do protisměru", "provoz převeden")
    if any(s in p for s in CIASTOCNE):
        return "lanes"
    return "full"


def nemodelovatelna(popis):
    """Obmedzenia, ktoré makroskopický model nevie vyjadriť.

    Zákaz odbočenia je obmedzenie pohybu v križovatke; model nemá odbočenia
    ani ich penalizácie, takže by sa musel modelovať ako uzavretie celej
    cesty — čo je niečo úplne iné.
    """
    p = (popis or "").lower()
    return "zákaz levého odbočení" in p or "zákaz pravého odbočení" in p

## 6 · Beh scenárov

In [6]:
def spusti_scenar(linky, rezim="full"):
    ls = []
    for l in linky:
        n = max(int(float(l["lanes"] or 2)), 1)
        ls.append({"link_id": l["link_id"], "direction": "both",
                   "closure_type": rezim, "lanes": n,
                   "lanes_remaining": (n - 1 if rezim == "lanes" and n > 1 else 0)})
    req = urllib.request.Request(f"{API_URL}/api/scenarios/run",
                                 data=json.dumps({"links": ls}).encode(),
                                 headers={"Content-Type": "application/json"})
    jid = json.load(urllib.request.urlopen(req, timeout=120))["id"]
    while True:
        st = api(f"/api/scenarios/{jid}/status")
        if st["status"] == "done":
            break
        if st["status"] == "error":
            raise RuntimeError(st.get("error"))
        time.sleep(2)
    res = api(f"/api/scenarios/{jid}/results")
    df = pd.DataFrame([{"link_id": int(f["properties"]["link_id"]),
                        "name": f["properties"].get("name"),
                        "delta": float(f["properties"].get("delta_vol") or 0)}
                       for f in res["features"]])
    return df, st.get("delta_summary", {})

VYSLEDKY, PRESKOCENE = {}, []

for _, row in kand.sort_values("valid_from").iterrows():
    label = f"{row.id} · {row.street_name or '(bez názvu)'}"
    if nemodelovatelna(row.description_cs):
        PRESKOCENE.append({"uzávierka": label, "liniek": 0, "objem": 0,
                           "dôvod": "zákaz odbočenia — model nemá odbočenia"})
        print(f"⚠ {label}: zákaz odbočenia, model to nevie vyjadriť — preskakujem")
        continue

    zavrete, sposob = linky_uzavierky(row)
    objem = sum(l["vol"] for l in zavrete)
    rezim = typ_uzavierky(row.description_cs)

    if not zavrete or objem < 500:
        PRESKOCENE.append({"uzávierka": label, "liniek": len(zavrete),
                           "objem": objem, "dôvod": "model tu nemá dopravu"})
        print(f"⚠ {label}: {objem:,.0f} voz/deň — preskakujem")
        continue

    model, summary = spusti_scenar(zavrete, rezim)
    zid = {l["link_id"] for l in zavrete}
    ulice = (model[~model.link_id.isin(zid) & model.name.notna()]
             .groupby("name").delta.sum().sort_values(ascending=False))

    # Popis uzávierky spomína aj ulicu, ktorá je zavretá — extraktor ju
    # vytiahne spolu s obchádzkou. Tá prirodzene dopravu STRÁCA, takže by
    # skončila na konci poradia a pokazila štatistiku. Vyhadzujeme ju podľa
    # definície uzávierky, nie podľa výsledku modelu (to by bolo kruhové).
    zavrete_mena = {norm(l["name"]) for l in zavrete if l["name"]}
    zavrete_mena |= {norm(row.street_name)} if isinstance(row.street_name, str) else set()
    obch = [u for u in row.obchadzka if norm(u) not in zavrete_mena]
    vyhodene = [u for u in row.obchadzka if norm(u) in zavrete_mena]

    VYSLEDKY[label] = {"row": row, "zavrete": zavrete, "sposob": sposob,
                       "rezim": rezim, "objem": objem, "model": model,
                       "ulice": ulice, "summary": summary,
                       "obchadzka": obch, "obchadzka_vyhodene": vyhodene}
    print(f"✓ {label}: {len(zavrete)} liniek ({sposob}), {objem:,.0f} voz/deň, "
          f"režim {rezim}, Σ|Δ| {summary.get('sum_abs_delta_vol',0):,.0f}"
          + (f"  [z obchádzky vyhodené ako zavreté: {', '.join(vyhodene)}]" if vyhodene else ""))

if PRESKOCENE:
    print()
    display(pd.DataFrame(PRESKOCENE))

⚠ 94340 · Ferrerova: 0 voz/deň — preskakujem
⚠ 94377 · Koperníkova: 0 voz/deň — preskakujem


✓ 94342 · Hlaváčova: 5 liniek (názov segmentu „Hlaváčova“), 21,215 voz/deň, režim full, Σ|Δ| 3,172,188
⚠ 94388 · Kosmákova: 0 voz/deň — preskakujem
⚠ 94402 · Viniční: 0 voz/deň — preskakujem


✓ 94415 · Olomoucká: 20 liniek (číslo cesty 430), 15,963 voz/deň, režim full, Σ|Δ| 2,808,768
⚠ 94413 · Skopalíkova: 0 voz/deň — preskakujem
⚠ 94434 · Vlárská: 0 voz/deň — preskakujem


✓ 94488 · Moravské náměstí: 31 liniek (názov segmentu „Moravské náměstí“), 100,503 voz/deň, režim lanes, Σ|Δ| 753,071


✓ 94493 · Jihlavská: 12 liniek (číslo cesty 602), 20,497 voz/deň, režim lanes, Σ|Δ| 186,075  [z obchádzky vyhodené ako zavreté: Jihlavská]


...

✓ 94511 · tunel Hlinky: 29 liniek (číslo cesty 42), 276,482 voz/deň, režim full, Σ|Δ| 13,851,825  [z obchádzky vyhodené ako zavreté: Žabovřeská, Bauerova]


✓ 94482 · Moravské náměstí: 31 liniek (názov segmentu „Moravské náměstí“), 100,503 voz/deň, režim lanes, Σ|Δ| 753,071


✓ 94460 · Sládkova: 6 liniek (najbližšie linky (bez zhody čísla/názvu)), 12,653 voz/deň, režim full, Σ|Δ| 3,220,746  [z obchádzky vyhodené ako zavreté: Merhautova]


✓ 94462 · VMO Dobrovského: 17 liniek (číslo cesty 42), 324,641 voz/deň, režim full, Σ|Δ| 11,880,014



,uzávierka,liniek,objem,dôvod
0,94340 · Ferrerova,2,0.0,model tu nemá dopravu
1,94377 · Koperníkova,14,0.0,model tu nemá dopravu
2,94388 · Kosmákova,7,0.0,model tu nemá dopravu
3,94402 · Viniční,11,0.0,model tu nemá dopravu
4,94413 · Skopalíkova,7,0.0,model tu nemá dopravu
5,94434 · Vlárská,11,0.0,model tu nemá dopravu


## 7 · Hlavný test — navrhol model tú istú obchádzku ako úrad? ⭐

Pre každú uzávierku zoberieme ulice z **oficiálnej obchádzky** a zistíme,
na akom mieste sú v modelovom poradí nárastu. Ulíc v sieti je ~2 060, takže
poradie v prvej desiatke znamená horné 0,5 %.

In [7]:
riadky, detail = [], []
for label, R in VYSLEDKY.items():
    poradie = list(R["ulice"].index)
    n = len(poradie)
    miesta = []
    for u in R["obchadzka"]:
        if u in poradie:
            p = poradie.index(u) + 1
            miesta.append(p)
            detail.append({"uzávierka": label, "obchádzková ulica": u,
                           "poradie v modeli": p, "z ulíc": n,
                           "percentil": round(100 * p / n, 1),
                           "Δ voz/deň": round(R["ulice"][u])})
    if not miesta:
        continue
    riadky.append({
        "uzávierka": label,
        "obchádzkových ulíc": len(miesta),
        "najlepšie poradie": min(miesta),
        "medián poradia": int(pd.Series(miesta).median()),
        "v TOP 10": sum(1 for p in miesta if p <= 10),
        "v TOP 5 %": sum(1 for p in miesta if p <= 0.05 * n),
        "z ulíc": n,
    })

OBCH = pd.DataFrame(riadky)
print("Súhrn — ako vysoko model umiestnil oficiálnu obchádzku:")
display(OBCH)
print("\nDetail po uliciach:")
display(pd.DataFrame(detail).sort_values(["uzávierka","poradie v modeli"]))

Súhrn — ako vysoko model umiestnil oficiálnu obchádzku:


,uzávierka,obchádzkových ulíc,najlepšie poradie,medián poradia,v TOP 10,v TOP 5 %,z ulíc
0,94342 · Hlaváčova,3,1914,1997,0,0,2059
1,94415 · Olomoucká,3,17,1973,0,1,2060
2,94488 · Moravské náměstí,5,27,1282,0,1,2059
3,94493 · Jihlavská,4,1,203,2,2,2060
4,94509 · VMO Dobrovského,7,8,21,1,5,2059
5,94502 · Královopolský tunel,4,24,241,0,2,2059
6,94507 · tunel Hlinky,9,8,50,1,6,2059
7,94417 · Olomoucká,3,17,1973,0,1,2060
8,94511 · tunel Hlinky,7,8,21,1,5,2059
9,94482 · Moravské náměstí,6,10,942,1,2,2059



Detail po uliciach:


,uzávierka,obchádzková ulica,poradie v modeli,z ulíc,percentil,Δ voz/deň
0,94342 · Hlaváčova,Rokytova,1914,2059,93.0,-131
1,94342 · Hlaváčova,Žarošická,1997,2059,97.0,-3005
2,94342 · Hlaváčova,Jedovnická,2047,2059,99.4,-20723
5,94415 · Olomoucká,Ostravská,17,2060,0.8,14870
3,94415 · Olomoucká,Těžební,1973,2060,95.8,-1003
4,94415 · Olomoucká,Průmyslová,2007,2060,97.4,-3748
37,94417 · Olomoucká,Ostravská,17,2060,0.8,14870
35,94417 · Olomoucká,Těžební,1973,2060,95.8,-1003
36,94417 · Olomoucká,Průmyslová,2007,2060,97.4,-3748
51,94460 · Sládkova,Křižíkova,44,2060,2.1,5953


In [8]:
# Tie isté stavby bývajú v NDIC vo viacerých záznamoch (94509/94511, 94415/94417)
# s rovnakou obchádzkou. Ako pozorovanie sa počítajú raz.
D = pd.DataFrame(detail)
if len(D):
    D = D.drop_duplicates(subset=["obchádzková ulica", "poradie v modeli"])

    # Dve rôzne otázky, dve rôzne metriky:
    #   najlepšia ulica -> trafil model KORIDOR obchádzky?
    #   medián cez ulice -> zreprodukoval CELÚ predpísanú trasu?
    # Predpísaná trasa býva zoznam 3-9 ulíc vrátane úsekov, ktoré dopravu
    # strácajú, takže medián je prísnejší, než na čo je model stavaný.
    G = (D.groupby("uzávierka")
           .agg(najlepsia=("percentil", "min"),
                median=("percentil", "median"),
                ulic=("percentil", "size"))
           .sort_values("najlepsia"))

    print(f"obchádzkových ulíc {len(D)} v {len(G)} uzávierkach "
          f"(náhoda by dala medián 50 %)\n")
    for col, otazka in [("najlepsia", "trafil koridor obchádzky"),
                        ("median",    "zreprodukoval celú trasu")]:
        v = G[col]
        riadok = (f"{otazka:<26} medián {v.median():>5.1f} % | "
                  f"TOP 5 % {int((v<=5).sum())}/{len(v)} | "
                  f"TOP 10 % {int((v<=10).sum())}/{len(v)}")
        try:
            from scipy.stats import wilcoxon
            _, p = wilcoxon(v - 50, alternative="less")
            riadok += f" | p = {p:.4f}"
        except Exception:
            pass
        print(riadok)

    print("\nPo uzávierkach:")
    display(G.round(1))
    print("Percentil 0 % = 1. miesto z ~2 060 ulíc.")

obchádzkových ulíc 38 v 10 uzávierkach (náhoda by dala medián 50 %)



trafil koridor obchádzky   medián   0.7 % | TOP 5 % 9/10 | TOP 10 % 9/10 | p = 0.0020
zreprodukoval celú trasu   medián   6.0 % | TOP 5 % 5/10 | TOP 10 % 6/10 | p = 0.0410

Po uzávierkach:


,najlepsia,median,ulic
uzávierka,,,
94462 · VMO Dobrovského,0.0,0.5,2
94493 · Jihlavská,0.0,9.9,4
94507 · tunel Hlinky,0.4,2.0,8
94509 · VMO Dobrovského,0.4,1.0,7
94482 · Moravské náměstí,0.5,0.5,1
94415 · Olomoucká,0.8,95.8,3
94502 · Královopolský tunel,1.2,11.7,4
94488 · Moravské náměstí,1.3,62.3,5
94460 · Sládkova,2.1,2.1,1


Percentil 0 % = 1. miesto z ~2 060 ulíc.


## 8 · Kontrolný pohľad — Waze zápchy

Slabší test: Waze meria *zdržanie*, model *objem*. Ulica môže dostať autá
a nezapchať sa, ak má kapacitu. Berieme to ako doplnok, nie ako verdikt.

Pozorovaná zmena je **difference-in-differences** — zmena na ulici mínus
celomestská zmena, aby v tom nebola sezóna. Šumové dno je z **placebo okna**
(ten istý výpočet na dvoch obdobiach *pred* uzávierkou).

In [9]:
def na_den(a, b):
    v = jams[(jams.ts >= a) & (jams.ts < b) & jams.vsedny
             & (jams.quality_score.fillna(0) >= QUALITY_MIN) & jams.street_name.notna()]
    dni = max(v.den.nunique(), 1)
    g = v.groupby("street_name").agg(zapch=("id","size"), zdrz=("delay_seconds","median"))
    g["na_den"] = g.zapch / dni
    return g

def did(vf, vt):
    pred = na_den(vf - pd.Timedelta(days=OKNO_PRED_DNI), vf)
    pocas = na_den(vf + pd.Timedelta(days=NABEH_DNI),
                   min(vt, vf + pd.Timedelta(days=OKNO_POCAS_MAX)))
    s = pred.join(pocas, how="outer", lsuffix="_pred", rsuffix="_pocas").fillna(0)
    d = s[(s.zapch_pred >= MIN_ZAPCH) & (s.zapch_pocas >= MIN_ZAPCH)]
    k = ((d.na_den_pocas + .1) / (d.na_den_pred + .1)).median() if len(d) else 1.0
    s["d_zapch"] = s.na_den_pocas - s.na_den_pred * k
    s.attrs["kontrola"] = k
    return s

def prah_sumu(vf):
    a = na_den(vf - pd.Timedelta(days=2*OKNO_PRED_DNI), vf - pd.Timedelta(days=OKNO_PRED_DNI))
    b = na_den(vf - pd.Timedelta(days=OKNO_PRED_DNI), vf)
    s = a.join(b, how="outer", lsuffix="_a", rsuffix="_b").fillna(0)
    d = s[(s.zapch_a >= MIN_ZAPCH) & (s.zapch_b >= MIN_ZAPCH)]
    k = ((d.na_den_b + .1) / (d.na_den_a + .1)).median() if len(d) else 1.0
    return (s.na_den_b - s.na_den_a * k).abs().quantile(0.95)

for label, R in VYSLEDKY.items():
    vf, vt = R["row"].valid_from, R["row"].valid_to
    # Waze dáta začínajú 14. 4. 2026; staršie uzávierky nemajú okno "pred".
    if pd.isna(vf) or pd.isna(vt) or vf < jams.ts.min() + pd.Timedelta(days=2*OKNO_PRED_DNI):
        continue
    R["waze"] = did(vf, vt)
    R["prah"] = prah_sumu(vf)

riadky = []
for label, R in VYSLEDKY.items():
    if "waze" not in R:
        continue
    w = R["waze"]
    w = w[(w.zapch_pred >= MIN_ZAPCH) | (w.zapch_pocas >= MIN_ZAPCH)].d_zapch.sort_values(ascending=False)
    idx = {norm(i): i for i in w.index}
    potvrd = sum(1 for u in R["ulice"].head(TOP_N).index
                 if norm(u) in idx and w[idx[norm(u)]] > R["prah"])
    obch_potvrd = sum(1 for u in R["obchadzka"]
                      if norm(u) in idx and w[idx[norm(u)]] > R["prah"])
    riadky.append({"uzávierka": label,
                   "kontrolný faktor": round(w.attrs.get("kontrola", 1) if hasattr(w,'attrs') else R["waze"].attrs["kontrola"], 2),
                   "prah šumu": round(R["prah"], 2),
                   f"model TOP-{TOP_N} potvrdené": f"{potvrd}/{TOP_N}",
                   "obchádzka potvrdená": f"{obch_potvrd}/{len(R['obchadzka'])}",
                   "TOP 3 podľa Waze": ", ".join(w.head(3).index)})
display(pd.DataFrame(riadky))

,uzávierka,kontrolný faktor,prah šumu,model TOP-10 potvrdené,obchádzka potvrdená,TOP 3 podľa Waze
0,94511 · tunel Hlinky,1.08,1.54,5/10,2/7,"Husova, Údolní, Bayerova"
1,94482 · Moravské náměstí,1.01,1.70,1/10,1/6,"Údolní, Bayerova, Husova"
2,94460 · Sládkova,0.93,1.72,1/10,0/1,"Nádražní, Vejrostova, Dornych"
3,94462 · VMO Dobrovského,0.76,1.88,0/10,0/2,"Rybkova, 3945, Tetčická"


## 9 · Nadhodnocuje model presun?

Model nemá elasticitu dopytu — nikto cestu nezruší ani neprejde na MHD.
Z toho **vyplýva predpoveď**, že modelový efekt bude systematicky väčší než
pozorovaný. Je to overiteľné tvrdenie o vlastnej známej slabine.

In [10]:
riadky = []
for label, R in VYSLEDKY.items():
    if "waze" not in R:
        continue
    w = R["waze"]; idx = {norm(i): i for i in w.index}
    par = [(R["ulice"][u], float(w.loc[idx[norm(u)], "d_zapch"]))
           for u in R["ulice"].index if norm(u) in idx]
    par = [(a, b) for a, b in par if abs(a) > 100]
    if len(par) < 5:
        continue
    mm = sum(1 for a, _ in par if a > 0)
    ww = sum(1 for _, b in par if b > R["prah"])
    riadky.append({"uzávierka": label, "ulíc": len(par),
                   "model hlási nárast": mm, "Waze potvrdí": ww,
                   "pomer": round(ww / max(mm, 1), 2)})
if riadky:
    display(pd.DataFrame(riadky))
    print("Pomer < 1 podporuje hypotézu, že model presun nadhodnocuje.")

,uzávierka,ulíc,model hlási nárast,Waze potvrdí,pomer
0,94511 · tunel Hlinky,358,238,28,0.12
1,94482 · Moravské náměstí,234,144,27,0.19
2,94460 · Sládkova,301,191,20,0.10
3,94462 · VMO Dobrovského,346,224,6,0.03


Pomer < 1 podporuje hypotézu, že model presun nadhodnocuje.


## 10 · Export

In [11]:
import datetime as _dt

if len(OBCH):
    OBCH.to_csv(OUT / "obchadzky_suhrn.csv", index=False)
if len(D):
    D.to_csv(OUT / "obchadzky_detail.csv", index=False)

for label, R in VYSLEDKY.items():
    slug = "".join(c for c in norm(label) if c.isalnum() or c in " _-").replace(" ", "_")
    R["ulice"].head(50).to_frame("delta_voz_den").to_csv(OUT / f"model_{slug}.csv")

META = {
    "vytvorene": _dt.datetime.now().isoformat(timespec="seconds"),
    "nastavenie": {"RADIUS_M": RADIUS_M, "OKNO_PRED_DNI": OKNO_PRED_DNI,
                   "NABEH_DNI": NABEH_DNI, "OKNO_POCAS_MAX": OKNO_POCAS_MAX,
                   "MIN_ZAPCH": MIN_ZAPCH, "QUALITY_MIN": QUALITY_MIN},
    "uzavierky": {k: {"id": int(R["row"].id), "sposob_naparovania": R["sposob"],
                      "rezim": R["rezim"], "zavretych_liniek": len(R["zavrete"]),
                      "objem": round(R["objem"]), "obchadzka": R["obchadzka"]}
                  for k, R in VYSLEDKY.items()},
    "preskocene": PRESKOCENE,
}
(OUT / "meta.json").write_text(json.dumps(META, indent=1, ensure_ascii=False), encoding="utf-8")
print("uložené do", OUT.resolve())
for p in sorted(OUT.iterdir()):
    print("  ", p.name)

uložené do /Users/magdalena.ondruskova/git/personal/Traffic-sim-backend/simulation_for_article/updated_version_12/vyhodnotenie
   meta.json
   model_94327__zahumenice.csv
   model_94342__hlavacova.csv
   model_94347__ostravska.csv
   model_94348__ostravska.csv
   model_94381__drobneho.csv
   model_94382__nan.csv
   model_94384__drobneho.csv
   model_94386__drobneho.csv
   model_94404__frycajova.csv
   model_94415__olomoucka.csv
   model_94417__olomoucka.csv
   model_94439__terezy_novakove.csv
   model_94446__bohunicka.csv
   model_94448__bohunicka.csv
   model_94460__sladkova.csv
   model_94462__vmo_dobrovskeho.csv
   model_94476__jihlavska.csv
   model_94482__moravske_namesti.csv
   model_94485__bauerova.csv
   model_94488__moravske_namesti.csv
   model_94490__netroufalky.csv
   model_94491__na_piskove_ceste.csv
   model_94492__jihlavska.csv
   model_94493__jihlavska.csv
   model_94500__42.csv
   model_94502__kralovopolsky_tunel.csv
   model_94507__tunel_hlinky.csv
   model_94509__vmo